In [1]:
import sys
import plotly.graph_objects as go
from pathlib import Path
import pandas as pd
import numpy as np
import re
# --- Data Loading ---

files = [
    'data/prices_round_1_day_-1.csv',
    'data/prices_round_1_day_-2.csv',
    'data/prices_round_1_day_0.csv'
]

prices = []
for file in files:
    try:
        df = pd.read_csv(file, sep=';')
        day_match = re.search(r'day_(-?\d+)', file)
        if day_match:
            df['day'] = int(day_match.group(1))
        prices.append(df)
    except FileNotFoundError:
        print(f"Warning: {file} not found.")

df_total = pd.concat(prices, ignore_index=True)
df_total = df_total.sort_values(by=['day', 'timestamp']).reset_index(drop=True)

In [8]:
from sklearn.linear_model import LinearRegression

def calculate_book_vwap(df):
    nominal = 0
    volume = 0
    for i in range(1, 4):
        nominal += (df[f'bid_price_{i}'] * df[f'bid_volume_{i}']).fillna(0)
        nominal += (df[f'ask_price_{i}'] * df[f'ask_volume_{i}']).fillna(0)
        volume += df[f'bid_volume_{i}'].fillna(0)
        volume += df[f'ask_volume_{i}'].fillna(0)
    return nominal / volume

def linear_regression(df):
    df['VWAP'] = calculate_book_vwap(df)
    #Fills NaNs with the previous valid value
    df['VWAP'] = df['VWAP'].ffill()
    
    # If the VERY first row is NaN, ffill won't work, so we drop any remaining
    df_clean = df.dropna(subset=['VWAP'])
    y = df[['VWAP']]
    X = df['timestamp'].values.reshape(-1, 1)

    model = LinearRegression()
    model.fit(X,y)

    slope = model.coef_[0]
    
    # 2. Intercept (c): Where the line hits the y-axis when X is 0
    intercept = model.intercept_
    
    # 3. R-Squared: How well the VWAP actually explains the price
    r_squared = model.score(X, y)
    
    print(f"Slope (Beta): {slope}")
    print(f"Intercept: {intercept}")
    print(f"R^2: {r_squared}")
    return model

In [9]:
days = [-1,-2, 0]
for day in days:
        subset = df_total[(df_total['product'] == "INTARIAN_PEPPER_ROOT") & (df_total['day'] == day)].copy()
        if subset.empty:
            continue
        linear_regression(subset)

Slope (Beta): [0.00100008]
Intercept: [10999.98221496]
R^2: 0.9998751258021757
Slope (Beta): [0.00099994]
Intercept: [10000.05060334]
R^2: 0.999894073925284
Slope (Beta): [0.00099987]
Intercept: [12000.07174422]
R^2: 0.9998557652766036
